# MOMENT SAE — Sparse Autoencoder Interpretability
Train a ReLU SAE on layer 18 of MOMENT-1-large to find monosemantic features for frequency, trend, and noise.

In [ ]:
!pip install momentfm torch numpy matplotlib pandas pytest --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Set working directory to a folder on Drive so data/checkpoints persist across sessions
WORK_DIR = '/content/drive/MyDrive/moment_sae'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

## Setup
Upload or clone the project scripts to this directory:
`config.py`, `sae.py`, `generate_activations.py`, `train_sae.py`, `visualize.py`

Or clone from your repo:
```
!git clone <your-repo-url> . --quiet
```

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!pytest tests/ -v 2>&1 | tail -20

## Step 1: Extract Activations
Generates synthetic time series and runs them through frozen MOMENT-1-large.
Saves ~1.25 GB to `data/activations.npy`. Skip this cell if `data/activations.npy` already exists.

In [ ]:
import os
if os.path.exists('data/activations.npy'):
    print('Activations already exist — skipping extraction.')
else:
    %run generate_activations.py

## Step 2: Train SAE
Trains for 50,000 steps (~25–30 min on T4). Checkpoints saved every 10,000 steps.
Re-running this cell resumes from the latest checkpoint.

In [ ]:
%run train_sae.py

## Step 3: Visualize Features
Generates per-feature top-20 plots, summary dashboard, and selectivity table.
Results saved to `outputs/`.

In [ ]:
%run visualize.py

In [ ]:
import pandas as pd
table = pd.read_csv('outputs/selectivity_table.csv')
print(f'Total live features: {len(table)}')
print(f'Monosemantic candidates: {(table["dominant_axis"] != "none").sum()}')
print('\nTop monosemantic features by dominant axis:')
print(table[table['dominant_axis'] != 'none']
      .sort_values('freq_sel', ascending=False)
      .head(20)
      .to_string(index=False))

In [ ]:
from IPython.display import Image, display
import glob

plots = sorted(glob.glob('outputs/feature_*_top20.png'))
if plots:
    display(Image(plots[0]))
    print(f'Showing: {plots[0]}')